# Fine-tune IndoBERT untuk Sentiment MyTelkomsel v2 (Google Colab)

Pasangan dari `02_v2_baseline_tfidf_lr.ipynb` bagian 7. Notebook ini **tidak pernah melihat label gold**:
gold dikirim hanya sebagai teks, dan penilaiannya dilakukan di notebook utama.

**Langkah:**
1. Di notebook utama, jalankan sampai bagian 7. Itu membuat `data/bert_data.zip`.
2. Di Colab: *Runtime → Change runtime type → GPU (T4 cukup)*.
3. Jalankan semua sel di sini. Sel "Data" akan meminta upload `bert_data.zip`.
4. Di akhir, `indobert_results.zip` otomatis terunduh. Ekstrak isinya (3 file) ke folder `data/bert/` di proyek
   (menimpa hasil lama), lalu jalankan ulang notebook utama.

Waktu di T4: run 1 (4 epoch, ~7.500 ulasan) = 6,1 menit training, jadi ~1,5 menit per epoch.

In [ ]:
# Colab sudah memuat torch. transformers/accelerate dipasang agar versinya jelas.
!pip -q install -U transformers accelerate

In [ ]:
import json, os, time, random, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import f1_score

# ---------- KONFIGURASI ----------
MODEL_NAME = "indobenchmark/indobert-base-p1"   # alternatif: "indolem/indobertweet-base-uncased"
EPOCHS     = 2          # run 1 (4 epoch): eval_loss naik sejak epoch 1, F1 epoch 2->4 hanya +0.004
LR         = 2e-5
BATCH      = 32
MAX_LEN    = 128
SEED       = 42
PATIENCE   = 1          # berhenti jika macro-F1 val tidak naik 1 epoch
SAVE_MODEL_TO_DRIVE = False   # True = simpan bobot (~500 MB) ke Google Drive

# Uji cepat lokal (CPU, model kecil). Di Colab biarkan False.
SMOKE = os.environ.get("SMOKE_TEST") == "1"
if SMOKE:
    MODEL_NAME, EPOCHS, BATCH, MAX_LEN = os.environ.get("SMOKE_MODEL", MODEL_NAME), 1, 8, 32

LABELS = ["negative", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
WORK = Path(os.environ.get("WORK_DIR", "/content"))
IN_DIR, OUT_DIR = WORK / "bert", WORK / "bert_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu" and not SMOKE:
    print("PERINGATAN: tidak ada GPU. Ganti runtime ke GPU, kalau tidak training akan sangat lama.")

## Data

In [ ]:
zip_path = WORK / "bert_data.zip"
if not IN_DIR.exists():
    if not zip_path.exists():
        from google.colab import files          # hanya ada di Colab
        up = files.upload()                      # pilih data/bert_data.zip dari proyek
        zip_path = WORK / next(iter(up))
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(WORK)

train = pd.read_csv(IN_DIR / "train.csv")
val   = pd.read_csv(IN_DIR / "val.csv")
gold  = pd.read_csv(IN_DIR / "gold_texts.csv", dtype={"reviewId": str})   # TANPA label
manifest = json.loads((IN_DIR / "manifest.json").read_text())
if SMOKE:
    train = train.groupby("label").head(32); val = val.groupby("label").head(16); gold = gold.head(20)
for name, d in [("train", train), ("val", val)]:
    print(name, len(d), d["label"].value_counts().to_dict())
print("gold (teks saja):", gold["split"].value_counts().to_dict())
print("manifest:", manifest)
assert set(train["label"]) <= set(LABELS) and "label" not in gold.columns

## Fine-tune

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tok([str(t) for t in texts], truncation=True, max_length=MAX_LEN)
        self.y = None if labels is None else [LABEL2ID[l] for l in labels]
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.y is not None:
            item["labels"] = self.y[i]
        return item

# Bobot kelas (data latih ~80% negatif)
counts = train["label"].value_counts()
class_w = torch.tensor([len(train) / (len(LABELS) * counts[l]) for l in LABELS], dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=class_w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    return {"macro_f1": f1_score(p.label_ids, np.argmax(p.predictions, -1), average="macro")}

steps_per_epoch = int(np.ceil(len(train) / BATCH))
args = TrainingArguments(
    output_dir=str(OUT_DIR / "ckpt"), num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, weight_decay=0.01,
    warmup_steps=int(0.06 * steps_per_epoch * EPOCHS),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    fp16=(DEVICE == "cuda"), report_to="none", seed=SEED, logging_steps=50)
trainer = WeightedTrainer(
    model=model, args=args, train_dataset=DS(train["text"], train["label"]),
    eval_dataset=DS(val["text"], val["label"]), compute_metrics=compute_metrics,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
hist = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
print(pd.DataFrame(hist)[["epoch", "eval_loss", "eval_macro_f1"]])
print(f"Waktu training: {train_minutes:.1f} menit | checkpoint terbaik: {trainer.state.best_model_checkpoint}")

## Prediksi gold (teks saja) dan simpan hasil

In [ ]:
def predict_proba(texts):
    logits = trainer.predict(DS(texts)).predictions
    e = np.exp(logits - logits.max(-1, keepdims=True))
    return (e / e.sum(-1, keepdims=True))[:, LABEL2ID["positive"]]

gold["proba_pos"] = predict_proba(gold["text"])
gold[["reviewId", "split", "proba_pos"]].to_csv(OUT_DIR / "preds_indobert.csv", index=False)
# Probabilitas di VAL rating (urutan sama dengan val.csv) -> notebook utama memilih threshold di sini, bukan di gold
pd.DataFrame({"row": range(len(val)), "proba_pos": predict_proba(val["text"])}).to_csv(
    OUT_DIR / "preds_val_rating.csv", index=False)

# Latency: 100 ulasan, batch 64, setelah warm-up
sample = gold["text"].tolist()[:100] * (1 if len(gold) >= 100 else 100 // max(len(gold), 1) + 1)
sample = sample[:100]
predict_proba(sample[:8])
if DEVICE == "cuda":
    torch.cuda.synchronize()
t0 = time.time(); predict_proba(sample)
if DEVICE == "cuda":
    torch.cuda.synchronize()
latency_100 = time.time() - t0

best = max(hist, key=lambda h: h["eval_macro_f1"])
info = {"model_name": MODEL_NAME, "model_version": f"indobert-v2-{time.strftime('%Y%m%d-%H%M')}-{manifest['data_hash'][:6]}",
        "transformers": transformers.__version__, "torch": torch.__version__,
        "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
        "epochs_max": EPOCHS, "best_epoch": best["epoch"], "val_rating_macro_f1_best": best["eval_macro_f1"],
        "lr": LR, "batch": BATCH, "max_len": MAX_LEN, "seed": SEED, "n_train": len(train), "n_val": len(val),
        "train_minutes": round(train_minutes, 2), "latency_100_sec": round(latency_100, 3),
        "smoke_test": SMOKE, "data_manifest": manifest, "eval_history": hist}
(OUT_DIR / "run_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps({k: v for k, v in info.items() if k != "eval_history"}, indent=2))

In [ ]:
res_zip = WORK / "indobert_results.zip"
with zipfile.ZipFile(res_zip, "w") as z:
    for f in ["preds_indobert.csv", "preds_val_rating.csv", "run_info.json"]:
        z.write(OUT_DIR / f, arcname=f)
print("Hasil:", res_zip)

if SAVE_MODEL_TO_DRIVE and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/mytsel_sentimen/indobert_v2")
    trainer.save_model(str(dst)); tok.save_pretrained(str(dst))
    print("Model disimpan ke", dst)

try:
    from google.colab import files
    files.download(str(res_zip))
except ImportError:
    pass   # bukan Colab (uji lokal)